# Consultas SQL · cuaderno del proyecto
**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [4]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [5]:
# Celda 2 · Cargar la muestra
ENLACE = "https://github.com/giseth18-dotcom/analisis-de-datos/blob/main/datos/originales/Bases%20de%20Datos%20General.csv"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

Cargué 16037 filas y 6 columnas.


,a_o,mes,departamento,ciudad,paisoeeresidencia,cant_extranjeros_no_residentes
0,2015,Ene,Antioquia,Medellín,Albania,1
1,2015,Ene,Antioquia,Medellín,Alemania,394
2,2015,Ene,Antioquia,Medellín,Angola,1
3,2015,Ene,Antioquia,Medellín,Antillas Neerlandesas,10
4,2015,Ene,Antioquia,Medellín,Arabia Saudita,2


In [6]:
sql("""
SELECT COUNT(*) AS total_filas,
       COUNT(cant_extranjeros_no_residentes) AS con_dato,
       MIN(cant_extranjeros_no_residentes) AS minimo,
       MAX(cant_extranjeros_no_residentes) AS maximo
FROM datos
""")

,total_filas,con_dato,minimo,maximo
0,16037,16037,1,40247


## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [7]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

,filas
0,16037


## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [9]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

['a_o',
 'mes',
 'departamento',
 'ciudad',
 'paisoeeresidencia',
 'cant_extranjeros_no_residentes']

In [10]:
sql("""
SELECT *
FROM datos
WHERE paisoeeresidencia = 'Estados Unidos'
""")

,a_o,mes,departamento,ciudad,paisoeeresidencia,cant_extranjeros_no_residentes
0,2015,Ene,Antioquia,Medellín,Estados Unidos,4125
1,2015,Feb,Antioquia,Medellín,Estados Unidos,4760
2,2015,Mar,Antioquia,Medellín,Estados Unidos,5456
3,2015,Abr,Antioquia,Medellín,Estados Unidos,4820
4,2015,May,Antioquia,Medellín,Estados Unidos,5693
...,...,...,...,...,...,...
133,2026,Feb,Antioquia,Medellín,Estados Unidos,29074
134,2026,Mar,Antioquia,Medellín,Estados Unidos,34190
135,2026,Abr,Antioquia,Medellín,Estados Unidos,28272
136,2026,May,Antioquia,Medellín,Estados Unidos,27708


---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [11]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

,filas_total
0,16037


In [24]:
# Celda 6 · Consulta 1b: filas después del filtro
sql("""
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE mes IN ('Jul','Ago', 'Nov', 'Dic')

""")

,filas_filtradas
0,5230


In [26]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas
sql("""
SELECT mes, paisoeeresidencia,	cant_extranjeros_no_residentes
FROM datos
WHERE mes IN ('Jul','Ago', 'Nov', 'Dic')
LIMIT 15
""")

,mes,paisoeeresidencia,cant_extranjeros_no_residentes
0,Jul,Alemania,449
1,Jul,Andorra,2
2,Jul,Angola,2
3,Jul,Antillas Neerlandesas,64
4,Jul,Arabia Saudita,2
5,Jul,Argelia,1
6,Jul,Argentina,674
7,Jul,Aruba,32
8,Jul,Australia,164
9,Jul,Austria,51


**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [28]:
# Celda 8 · Consulta 2: resumen por grupo
sql("""
SELECT mes, COUNT(*) AS filas
FROM datos
GROUP BY mes
ORDER BY filas DESC
""")

,mes,filas
0,Mar,1458
1,Ene,1404
2,Feb,1393
3,Nov,1391
4,Dic,1352
5,Jun,1348
6,May,1310
7,Abr,1306
8,Sep,1299
9,Oct,1289


In [20]:
# Celda 8 · Consulta 2.1: resumen por grupo
sql("""
SELECT mes, SUM(cant_extranjeros_no_residentes) AS personas
FROM datos
GROUP BY mes
ORDER BY personas DESC
""")

,mes,personas
0,Dic,592314.0
1,Ene,558144.0
2,Feb,555870.0
3,Jul,546996.0
4,Mar,538824.0
5,Nov,529307.0
6,Ago,520352.0
7,Jun,515618.0
8,May,503807.0
9,Abr,495494.0


In [21]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT SUM(filas) AS suma_de_los_grupos
FROM (
    SELECT mes, COUNT(*) AS filas
    FROM datos
    GROUP BY mes
)
""")

,suma_de_los_grupos
0,16037.0


## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.